# 11 · Mixed-integer programming with SciPy (and Gurobi)

**Goal:** model discrete decisions with binary variables, solve with `scipy.optimize.milp` (HiGHS branch-and-bound), and **validate by brute force or dynamic programming** on small instances. Then build the same model in Gurobi.

Two classic modeling patterns:
* **Knapsack:** $\max v^\top x$ s.t. $w^\top x\le W$, $x\in\{0,1\}^n$.
* **Fixed charge (big-M linking):** flow $x_{ij}$ may use arc $(i,j)$ only if it is opened ($y_{ij}=1$), via $x_{ij}\le u_{ij}y_{ij}$, with cost $\sum c_{ij}x_{ij}+f_{ij}y_{ij}$.

Network flow LPs have integral vertices (notebook 08). Fixed charges break that structure, which is why they need branch-and-bound.

In [ ]:
import itertools
import numpy as np
from scipy.optimize import milp, linprog, LinearConstraint, Bounds

## 1. Knapsack: MILP vs dynamic programming

In [ ]:
values = np.array([10, 13, 7, 8, 9, 4, 12, 6], float)
weights = np.array([5, 7, 3, 4, 5, 2, 6, 3], float)
W = 15

res = milp(-values, constraints=LinearConstraint(weights[None, :], -np.inf, W),
           integrality=np.ones(len(values)), bounds=Bounds(0, 1))
x = np.round(res.x).astype(int)
print("MILP value:", -res.fun, " items:", np.nonzero(x)[0])

def knapsack_dp(v, w, cap):
    best = np.zeros(cap + 1)
    for vi, wi in zip(v, w.astype(int)):
        for r in range(cap, wi - 1, -1):          # iterate backwards: each item at most once
            best[r] = max(best[r], best[r - wi] + vi)
    return best[cap]

print("DP value:  ", knapsack_dp(values, weights, W))
assert np.isclose(-res.fun, knapsack_dp(values, weights, W))

# the LP relaxation gives an upper bound, which is the quantity branch-and-bound prunes with
lp = linprog(-values, A_ub=weights[None, :], b_ub=[W], bounds=(0, 1), method="highs")
print("LP relaxation bound:", -lp.fun)
assert -lp.fun >= -res.fun - 1e-9

## 2. Fixed-charge network design

In [ ]:
# (i, j, unit cost c, capacity u, fixed opening cost f); ship 10 units from 0 to 4
ARCS = [(0, 1, 1, 8, 10), (0, 2, 3, 10, 4), (1, 2, 1, 5, 3), (1, 3, 2, 8, 12),
        (2, 3, 1, 10, 6), (2, 4, 4, 10, 8), (3, 4, 1, 10, 9), (1, 4, 6, 4, 2)]
n, s, t, demand = 5, 0, 4, 10
m = len(ARCS)
cc = np.array([a[2] for a in ARCS], float)
uu = np.array([a[3] for a in ARCS], float)
ff = np.array([a[4] for a in ARCS], float)
Ninc = np.zeros((n, m))
for k, (i, j, *_) in enumerate(ARCS):
    Ninc[i, k] = 1; Ninc[j, k] = -1
rhs = np.zeros(n); rhs[s] = demand; rhs[t] = -demand

# variable vector z = [x (m flows), y (m binaries)]
obj = np.r_[cc, ff]
flow_cons = LinearConstraint(np.c_[Ninc, np.zeros((n, m))], rhs, rhs)
link = LinearConstraint(np.c_[np.eye(m), -np.diag(uu)], -np.inf, 0)      # x - u*y <= 0
integrality = np.r_[np.zeros(m), np.ones(m)]
bnds = Bounds(np.zeros(2 * m), np.r_[uu, np.ones(m)])
res = milp(obj, constraints=[flow_cons, link], integrality=integrality, bounds=bnds)
xf, yf = res.x[:m], np.round(res.x[m:])
print("MILP cost:", res.fun)
for k, (i, j, *_) in enumerate(ARCS):
    if yf[k] > 0.5:
        print(f"  open {i}->{j}, flow {xf[k]:.1f}")

**Validate by enumeration.** With $m=8$ arcs there are only $2^8=256$ designs. For each one, solve the flow LP on the open arcs and add the fixed costs. Brute force on toy instances is the most reliable way to catch modeling errors.

In [ ]:
best = np.inf
for ybits in itertools.product([0, 1], repeat=m):
    yv = np.array(ybits, float)
    lp = linprog(cc, A_eq=Ninc, b_eq=rhs, bounds=list(zip(np.zeros(m), uu * yv)), method="highs")
    if lp.status == 0:
        best = min(best, lp.fun + ff @ yv)
print("brute-force optimum:", best)
assert np.isclose(best, res.fun)

## 3. The same knapsack in Gurobi (optional)

`gurobipy` with your academic license (`pip install gurobipy`, then `grbgetkey <key>`). Gurobi models read closer to the math. The cell skips cleanly if Gurobi isn't installed.

In [ ]:
try:
    import gurobipy as gp
    from gurobipy import GRB

    mdl = gp.Model("knapsack")
    mdl.Params.OutputFlag = 0
    xg = mdl.addVars(len(values), vtype=GRB.BINARY, name="x")
    mdl.setObjective(gp.quicksum(values[i] * xg[i] for i in range(len(values))), GRB.MAXIMIZE)
    mdl.addConstr(gp.quicksum(weights[i] * xg[i] for i in range(len(values))) <= W, name="cap")
    mdl.optimize()
    print("Gurobi value:", mdl.ObjVal)
    assert np.isclose(mdl.ObjVal, -milp(-values, constraints=LinearConstraint(weights[None, :], -np.inf, W),
                                           integrality=np.ones(len(values)), bounds=Bounds(0, 1)).fun)
except ImportError:
    print("gurobipy not installed: skipping (pip install gurobipy)")
except Exception as e:                  # e.g. license not activated
    print("Gurobi not usable here:", e)

## Exercises

1. Port the fixed-charge model to Gurobi and add the **strong linking** $x_{ij}\le\min(u_{ij},\text{demand})\,y_{ij}$. Compare LP-relaxation bounds with the weak version.
2. Add a constraint "at most 3 arcs opened". How does the solution change?
3. Write a tiny branch-and-bound for knapsack yourself (LP bound + best-first search) and count nodes explored.